In [20]:
import cobra
from cobra.util.array import create_stoichiometric_matrix
from cobra.flux_analysis import flux_variability_analysis, gapfill
import cobra.flux_analysis.gapfilling as gf

import numpy as np
import pandas as pd

In [3]:
# Load the Human-GEM model
model = cobra.io.read_sbml_model("../data/Human-GEM.xml")
original_model = cobra.io.read_sbml_model("../data/Human-GEM.xml")

In [4]:
# Load the medium file 
rpmi1640 = pd.read_csv("../../media/RPMI1640.tsv", sep="\t")
fbs = pd.read_csv("../../media/FBS.tsv", sep="\t")   
combined = pd.concat([rpmi1640, fbs], ignore_index=True)

group_cols = ['Mets_Recon3D', 'Mets_HumanGEM', 'ExRxns_Recon3D', 'ExRxns_HumanGEM']
medium = combined.groupby(group_cols, as_index=False)['Concentration_M'].sum()

# print(medium.head())

# Conc2Rate 
# def conc_to_flux(conc_M, cell_conc, cell_weight_g=400e-12, doubling_time_h=24):
#     return conc_M / (cell_conc * cell_weight_g * doubling_time_h * 1000)

# medium["Flux_mmol_gDW_h"] = conc_to_flux(medium["Concentration_M"], 1e5)


# Create a dictionary for medium fluxes
medium_fluxes = {
    row["ExRxns_HumanGEM"]: -1000.0
    for _, row in medium.iterrows()
}

mandatory_ex_rxns = {
    "MAR09048": -1000.0,  # O2
    "MAR09047": -1000.0,  # H2O
    "MAR09058": -1000.0,  # CO2
    "MAR09079": -1000.0,  # H+
}

medium_fluxes.update(mandatory_ex_rxns)

medium_fluxes

{'MAR08386': -1000.0,
 'MAR09363': -1000.0,
 'MAR09269': -1000.0,
 'MAR09066': -1000.0,
 'MAR09062': -1000.0,
 'MAR09070': -1000.0,
 'MAR09273': -1000.0,
 'MAR09109': -1000.0,
 'MAR09082': -1000.0,
 'MAR09083': -1000.0,
 'MAR09285': -1000.0,
 'MAR09150': -1000.0,
 'MAR09460': -1000.0,
 'MAR09293': -1000.0,
 'MAR09146': -1000.0,
 'MAR09034': -1000.0,
 'MAR09063': -1000.0,
 'MAR09071': -1000.0,
 'MAR09067': -1000.0,
 'MAR09351': -1000.0,
 'MAR09078': -1000.0,
 'MAR09038': -1000.0,
 'MAR09039': -1000.0,
 'MAR09361': -1000.0,
 'MAR09081': -1000.0,
 'MAR09040': -1000.0,
 'MAR09041': -1000.0,
 'MAR09042': -1000.0,
 'MAR09077': -1000.0,
 'MAR09378': -1000.0,
 'MAR09043': -1000.0,
 'MAR09072': -1000.0,
 'MAR09145': -1000.0,
 'MAR09393': -1000.0,
 'MAR09068': -1000.0,
 'MAR09396': -1000.0,
 'MAR09144': -1000.0,
 'MAR09147': -1000.0,
 'MAR09143': -1000.0,
 'MAR09069': -1000.0,
 'MAR09074': -1000.0,
 'MAR09159': -1000.0,
 'MAR09044': -1000.0,
 'MAR09045': -1000.0,
 'MAR09429': -1000.0,
 'MAR09064

In [5]:
S = create_stoichiometric_matrix(model)

exchange_rxns = []
for i, rxn in enumerate(model.reactions):
    if np.count_nonzero(S[:, i]) == 1:
        exchange_rxns.append(rxn)

print(len(exchange_rxns), "exchange reactions found")

# for rxn in exchange_rxns:
#     print(rxn.id, rxn.name)

1660 exchange reactions found


In [14]:
# Find exchange reactions 
ex_rxns = model.exchanges # Warning find only exchange (not demand or sink if Recon3D)

# Apply medium constraints to the model 
for rxn in exchange_rxns:
    reaction = model.reactions.get_by_id(rxn.id)
    if rxn.id in medium_fluxes:
        # print(rxn.name)
        # reaction.lower_bound = medium_fluxes[rxn.id]
        reaction.lower_bound = -1000.0
    else:
        model.remove_reactions([reaction])

/Users/hugues_esc/.local/share/virtualenvs/2FASTC-97Eiqq48/lib/python3.11/site-packages/cobra/core/group.py:147: UserWarning: need to pass in a list
  warn("need to pass in a list")


In [7]:
# biomass_rxn = model.reactions.get_by_id('MAR04413')  
biomass_rxn = model.reactions.get_by_id('MAR13082')

# for met in biomass_rxn.metabolites: 
#     print(met.id, met.name)

In [27]:
pool_id = 'MAM10013c'  # protein_pool_biomass
protein_pool = model.metabolites.get_by_id(pool_id)

for rxn in protein_pool.reactions:
    if protein_pool in rxn.products:  # or rxn.metabolites[protein_pool] > 0
        print(rxn.id, "produces", pool_id)
        print(rxn.reaction)

MAR10062 produces MAM10013c
0.0721 MAM02006c + 0.0801 MAM02335c + 0.0512 MAM02340c + 0.0375 MAM02341c + 0.0556 MAM02342c + 0.0183 MAM02351c + 0.0428 MAM02376c + 0.0783 MAM02377c + 0.0228 MAM02380c + 0.0442 MAM02401c + 0.0911 MAM02404c + 0.0719 MAM02405c + 0.0222 MAM02408c + 0.0368 MAM02412c + 0.051 MAM02415c + 0.0661 MAM02416c + 0.0535 MAM02419c + 0.0098 MAM02420c + 0.0281 MAM02421c + 0.0667 MAM02423c --> 0.0801 MAM03063c + 0.0512 MAM03064c + 0.0375 MAM03065c + 0.0556 MAM03066c + 0.0183 MAM03067c + 0.0428 MAM03068c + 0.0783 MAM03069c + 0.0721 MAM03070c + 0.0228 MAM03071c + 0.0442 MAM03072c + 0.0911 MAM03073c + 0.0719 MAM03074c + 0.0222 MAM03075c + 0.0368 MAM03076c + 0.051 MAM03077c + 0.0661 MAM03078c + 0.0535 MAM03079c + 0.0098 MAM03080c + 0.0281 MAM03081c + 0.0667 MAM03082c + MAM10013c


In [15]:
model.objective = 'MAR13082'
original_model.objective = 'MAR13082'  # Biomass reaction
print(model.optimize().objective_value)
print(original_model.optimize().objective_value)

0.0
124.86814837744573


In [31]:
universal = original_model.copy()
for rxn in universal.exchanges:
    rxn.lower_bound = -1000
    rxn.upper_bound = 1000

solution = gf.GapFiller(model, universal, demand_reactions = False, integer_threshold=1e-9)
reactions = solution.fill()
for reaction in reactions[0]:
    print("Suggested:", reaction.id)

Read LP format model from file /var/folders/hd/x2nmhnb55t3_s_kjs_c8js6m0000gn/T/tmpx43s9kti.lp
Reading time = 0.04 seconds
: 8455 rows, 25942 columns, 110884 nonzeros
Read LP format model from file /var/folders/hd/x2nmhnb55t3_s_kjs_c8js6m0000gn/T/tmpa5zzudrn.lp
Reading time = 0.04 seconds
: 8455 rows, 22728 columns, 107670 nonzeros
Read LP format model from file /var/folders/hd/x2nmhnb55t3_s_kjs_c8js6m0000gn/T/tmpf6_yt64q.lp
Reading time = 0.04 seconds
: 8455 rows, 25942 columns, 110884 nonzeros


RuntimeError: Failed to validate gap filled model, try lowering the integer threshold.

In [26]:
for i, entries in enumerate(solution.fill()):
    print("---- Run %d ----" % (i + 1))
    for e in entries:
        print(e.id)

RuntimeError: Failed to validate gap filled model, try lowering the integer threshold.

In [23]:
for reaction in reactions[0]:
    print("Suggested:", reaction.name)

Suggested: Exchange of VLDL remnant
Suggested: Exchange of heme
Suggested: Exchange of alpha-tocotrienol
Suggested: Exchange of lipoic acid
Suggested: Exchange of biomass


In [16]:
i = 0
for rxn in exchange_rxns:
    # print(rxn.id, rxn.lower_bound, rxn.upper_bound, rxn.name)
    if rxn.lower_bound < 0.0:
        print(rxn.id, rxn.lower_bound, rxn.upper_bound, rxn.name)
        # print(rxn.id, rxn.name)
        i += 1
print(i)

MAR09034 -1000.0 1000.0 Exchange of glucose
MAR09038 -1000.0 1000.0 Exchange of histidine
MAR09039 -1000.0 1000.0 Exchange of isoleucine
MAR09040 -1000.0 1000.0 Exchange of leucine
MAR09041 -1000.0 1000.0 Exchange of lysine
MAR09042 -1000.0 1000.0 Exchange of methionine
MAR09043 -1000.0 1000.0 Exchange of phenylalanine
MAR09044 -1000.0 1000.0 Exchange of threonine
MAR09045 -1000.0 1000.0 Exchange of tryptophan
MAR09046 -1000.0 1000.0 Exchange of valine
MAR09047 -1000.0 1000.0 Exchange of H2O
MAR09048 -1000.0 1000.0 Exchange of O2
MAR09058 -1000.0 1000.0 Exchange of CO2
MAR09062 -1000.0 1000.0 Exchange of asparagine
MAR09063 -1000.0 1000.0 Exchange of glutamine
MAR09064 -1000.0 1000.0 Exchange of tyrosine
MAR09066 -1000.0 1000.0 Exchange of arginine
MAR09067 -1000.0 1000.0 Exchange of glycine
MAR09068 -1000.0 1000.0 Exchange of proline
MAR09069 -1000.0 1000.0 Exchange of serine
MAR09070 -1000.0 1000.0 Exchange of aspartate
MAR09071 -1000.0 1000.0 Exchange of glutamate
MAR09072 -1000.0 1

In [15]:
fva_result = flux_variability_analysis(model, reaction_list=ex_rxns)

Set parameter Username
Academic license - for non-commercial use only - expires 2025-10-14
Read LP format model from file /var/folders/hd/x2nmhnb55t3_s_kjs_c8js6m0000gn/T/tmpyj5qyye5.lp
Reading time = 0.04 seconds
: 8456 rows, 25943 columns, 110887 nonzeros
Set parameter Username
Academic license - for non-commercial use only - expires 2025-10-14
Read LP format model from file /var/folders/hd/x2nmhnb55t3_s_kjs_c8js6m0000gn/T/tmpzvx_v6yr.lp
Reading time = 0.04 seconds
: 8456 rows, 25943 columns, 110887 nonzeros
Set parameter Username
Academic license - for non-commercial use only - expires 2025-10-14
Read LP format model from file /var/folders/hd/x2nmhnb55t3_s_kjs_c8js6m0000gn/T/tmpyarmo_js.lp
Reading time = 0.04 seconds
: 8456 rows, 25943 columns, 110887 nonzeros
Set parameter Username
Academic license - for non-commercial use only - expires 2025-10-14
Read LP format model from file /var/folders/hd/x2nmhnb55t3_s_kjs_c8js6m0000gn/T/tmpb2mbk4yl.lp
Reading time = 0.06 seconds
: 8456 rows,

In [16]:
print(fva_result.head())

          minimum  maximum
MAR00659      0.0      0.0
MAR07108      0.0      0.0
MAR07110      0.0      0.0
MAR07112      0.0      0.0
MAR07114      0.0      0.0


In [17]:
# Check if any reactions have minimum > maximum
invalid_bounds = fva_result[fva_result['minimum'] > fva_result['maximum']]
if not invalid_bounds.empty:
    print("Reactions with minimum > maximum:")
    print(invalid_bounds)
else:
    print("No reactions have minimum > maximum.")

# Find reactions with minimum < 0
negative_min = fva_result[fva_result['minimum'] < 0]
print(f"\nNumber of reactions with minimum < 0: {len(negative_min)}")
print("List of reactions with minimum < 0:")
print(negative_min.index.tolist())


No reactions have minimum > maximum.

Number of reactions with minimum < 0: 46
List of reactions with minimum < 0:
['MAR09034', 'MAR09038', 'MAR09039', 'MAR09040', 'MAR09041', 'MAR09042', 'MAR09043', 'MAR09044', 'MAR09045', 'MAR09046', 'MAR09047', 'MAR09048', 'MAR09058', 'MAR09062', 'MAR09063', 'MAR09064', 'MAR09066', 'MAR09067', 'MAR09068', 'MAR09069', 'MAR09070', 'MAR09071', 'MAR09072', 'MAR09074', 'MAR09075', 'MAR09078', 'MAR09079', 'MAR09083', 'MAR09143', 'MAR09144', 'MAR09145', 'MAR09146', 'MAR09147', 'MAR09150', 'MAR09159', 'MAR09273', 'MAR09285', 'MAR09293', 'MAR09351', 'MAR09361', 'MAR09363', 'MAR09378', 'MAR09393', 'MAR09396', 'MAR09429', 'MAR08386']


In [18]:
# Assume `fva_result` is your DataFrame and `flux_dict` is your dictionary

# Step 1: Extract reactions
flux_dict_reactions = set(medium_fluxes.keys())
negative_min_reactions = set(fva_result[fva_result['minimum'] < 0].index)

# Step 2: Compare sets
common_reactions = flux_dict_reactions & negative_min_reactions
only_in_flux_dict = flux_dict_reactions - negative_min_reactions
only_in_fva_result = negative_min_reactions - flux_dict_reactions

# Step 3: Print results
print(f"Number of common reactions: {len(common_reactions)}")
print(f"Common reactions:\n{sorted(common_reactions)}")

print(f"\nReactions only in flux_dict: {len(only_in_flux_dict)}")
print(f"{sorted(only_in_flux_dict)}")

print(f"\nReactions only in FVA result with negative min: {len(only_in_fva_result)}")
print(f"{sorted(only_in_fva_result)}")

Number of common reactions: 46
Common reactions:
['MAR08386', 'MAR09034', 'MAR09038', 'MAR09039', 'MAR09040', 'MAR09041', 'MAR09042', 'MAR09043', 'MAR09044', 'MAR09045', 'MAR09046', 'MAR09047', 'MAR09048', 'MAR09058', 'MAR09062', 'MAR09063', 'MAR09064', 'MAR09066', 'MAR09067', 'MAR09068', 'MAR09069', 'MAR09070', 'MAR09071', 'MAR09072', 'MAR09074', 'MAR09075', 'MAR09078', 'MAR09079', 'MAR09083', 'MAR09143', 'MAR09144', 'MAR09145', 'MAR09146', 'MAR09147', 'MAR09150', 'MAR09159', 'MAR09273', 'MAR09285', 'MAR09293', 'MAR09351', 'MAR09361', 'MAR09363', 'MAR09378', 'MAR09393', 'MAR09396', 'MAR09429']

Reactions only in flux_dict: 7
['MAR09077', 'MAR09081', 'MAR09082', 'MAR09109', 'MAR09153', 'MAR09269', 'MAR09460']

Reactions only in FVA result with negative min: 0
[]
